In [1]:
from pathlib import Path
import MDAnalysis as mda
from MDAnalysis.coordinates.XTC import XTCWriter

In [2]:
from tqdm import tqdm

def pdb_folder_to_trajectory(
    pdb_dir: str,
    out_top: Path = Path("topology.pdb"),
    out_traj: Path = Path("traj.xtc"),
    pattern: str = "*.pdb",
    sort_key=None,   # optional: custom sorter for filenames
    dt_ps: float = 1.0,  # optional timestep metadata
):
    pdb_dir = Path(pdb_dir)
    pdbs = list(pdb_dir.glob(pattern))
    if not pdbs:
        raise FileNotFoundError(f"No files matched {pattern} in {pdb_dir}")

    # Sort frames deterministically (lexicographic by default)
    pdbs = sorted(pdbs, key=sort_key) if sort_key else sorted(pdbs)

    # Use first PDB as topology
    u0 = mda.Universe(str(pdbs[0]))

    # Write topology reference (optional but usually useful)
    u0.atoms.write(out_top)

    n_atoms = u0.atoms.n_atoms

    # Choose writer by extension (or swap to DCDWriter)
    if out_traj.suffix.lower() == ".xtc":
        Writer = XTCWriter
    elif out_traj.suffix.lower() == ".dcd":
        from MDAnalysis.coordinates.DCD import DCDWriter
        Writer = DCDWriter
    else:
        raise ValueError("out_traj must end with .xtc or .dcd")

    # with Writer(out_traj, n_atoms=n_atoms) as W:
    with Writer(out_traj.as_posix(), n_atoms=n_atoms) as W:
        for i, pdb in tqdm(enumerate(pdbs), total=len(pdbs), desc="Processing PDBs"):
            u = mda.Universe(str(pdb))

            if u.atoms.n_atoms != n_atoms:
                raise ValueError(
                    f"Atom count mismatch at frame {i}: {pdb.name} "
                    f"has {u.atoms.n_atoms}, expected {n_atoms}"
                )

            # Copy coordinates into u0 and write a frame
            u0.atoms.positions = u.atoms.positions
            # (Optional) store time metadata for some formats
            try:
                u0.trajectory.ts.time = i * dt_ps
            except Exception:
                pass

            W.write(u0.atoms)

    return out_top, out_traj

In [3]:
ens_dir = Path('/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/ProtSCAPE_ensembles')

for trial_dir in ens_dir.iterdir():


    trial = trial_dir.name.split('_')[-1] 
    for system_dir in trial_dir.iterdir():
        system = system_dir.name
        print(f"Processing trial {trial}, system {system}...")

        pdb_dir = system_dir / 'generated_traj'

        in_pdb = pdb_dir / 'pred_topology.pdb'
        in_xtc = pdb_dir / 'pred_traj.xtc'

        out_pdb = system_dir / f'{system}.pdb'
        out_xtc = system_dir / f'{system}.xtc'

        !cp {in_pdb} {out_pdb}
        !cp {in_xtc} {out_xtc}



        

Processing trial 1, system 6h86_A...
Processing trial 1, system 6jv8_A...
Processing trial 1, system 6p5h_A...
Processing trial 1, system 7jfl_C...
Processing trial 1, system 7lp1_A...
Processing trial 2, system 6h86_A...
Processing trial 2, system 6jv8_A...
Processing trial 2, system 6p5h_A...
Processing trial 2, system 7jfl_C...
Processing trial 2, system 7lp1_A...
Processing trial 3, system 6h86_A...
Processing trial 3, system 6jv8_A...
Processing trial 3, system 6p5h_A...
Processing trial 3, system 7jfl_C...
Processing trial 3, system 7lp1_A...
Processing trial 4, system 6h86_A...
Processing trial 4, system 6jv8_A...
Processing trial 4, system 6p5h_A...
Processing trial 4, system 7jfl_C...
Processing trial 4, system 7lp1_A...
Processing trial 5, system 6h86_A...
Processing trial 5, system 6jv8_A...
Processing trial 5, system 6p5h_A...
Processing trial 5, system 7jfl_C...
Processing trial 5, system 7lp1_A...


In [6]:
import subprocess

ens_dir = Path('/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles')
systems = ['6h86_A', '6p5h_A', '6jv8_A', '7lp1_A', '7jfl_C']

for trial_dir in ens_dir.iterdir():

    trial = trial_dir.name.split('_')[-1] 
    for system in systems:

        system_dir = trial_dir / system
        system_dir.mkdir(exist_ok=True, parents=True)  # Ensure system directory exists


        subprocess.run(["mv", str(trial_dir / f"{system}.pdb"), str(system_dir / f"{system}.pdb")])
        subprocess.run(["mv", str(trial_dir / f"{system}.xtc"), str(system_dir / f"{system}.xtc")])


mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6h86_A.pdb': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6h86_A.xtc': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6p5h_A.pdb': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6p5h_A.xtc': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6jv8_A.pdb': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles/ATLAS_1K_trial_1/6jv8_A.xtc': No such file or directory
mv: cannot stat '/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/MDGen_ensembles

In [3]:
def pdb_models_to_xtc(
    pdb_file: str,
    out_xtc: str = "traj.xtc",
    out_pdb0: str = "frame0.pdb",
    selection: str = "all",
):
    u = mda.Universe(pdb_file)
    atoms = u.select_atoms(selection)

    # write first frame as topology PDB
    u.trajectory[0]
    atoms.write(out_pdb0)

    # write trajectory
    with mda.coordinates.XTC.XTCWriter(out_xtc.as_posix(), atoms.n_atoms) as W:
        for ts in u.trajectory:
            W.write(atoms)

In [4]:
import subprocess
ens_dir = Path('/nfs/roberts/pi/pi_sk2433/shared/ProtSCAPE_2026_MDSimulations/AlphaFLOW_ensembles')

for trial_dir in ens_dir.iterdir():

    trial = trial_dir.name.split('_')[-1] 
    for system_dir in trial_dir.iterdir():
        system = system_dir.name
        print(f"Processing trial {trial}, system {system}...")

        pdb_file = system_dir / f'{system}.pdb'
        pdb_full = system_dir / f'{system}_full.pdb'
        subprocess.run(["mv", str(pdb_file), str(pdb_full)])

        out_xtc = system_dir / f'{system}.xtc'

        pdb_models_to_xtc(pdb_full, out_xtc, pdb_file)
   

Processing trial 1, system 6h86_A...


/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/XTC.py:119: UserWarning: Reader has no dt information, set to 1.0 ps
  time = ts.time


Processing trial 1, system 6jv8_A...


/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/XTC.py:119: UserWarning: Reader has no dt information, set to 1.0 ps
  time = ts.time


Processing trial 1, system 6p5h_A...


...

/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/XTC.py:119: UserWarning: Reader has no dt information, set to 1.0 ps
  time = ts.time


Processing trial 5, system 7jfl_C...


/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/XTC.py:119: UserWarning: Reader has no dt information, set to 1.0 ps
  time = ts.time


Processing trial 5, system 7lp1_A...


/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(
/home/lcl35/.conda/envs/protscape/lib/python3.12/site-packages/MDAnalysis/coordinates/XTC.py:119: UserWarning: Reader has no dt information, set to 1.0 ps
  time = ts.time
